## How to use this notebook

Run notebooks 01 and 02 first, then `Run All` here. It groups same-name recipe rows into versions,
prints the comparison for Chicken Inasal, and shows which row "fastest" selects. For a plain
terminal version of the same flow, run `python examples/demo_answer.py` from the repository root.

# 05 · Same-name recipe versions

Rows that share an exact `recipe_name` but contain different ingredients and instructions are
**versions of the same dish**, not duplicates to delete. This notebook uses `src/dish_groups.py`
to group them, merge near-identical rows, and produce a deterministic comparison the user can
choose from before the RAG step.

Run notebooks 01 and 02 first so `data/processed/cleaned_recipes.parquet` has
`canonical_ingredients`. If the file is missing, the first code cell stops with a clear message
telling you to run those notebooks.


In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

path = ROOT / "data/processed/cleaned_recipes.parquet"
if not path.exists():
    raise SystemExit("Missing data/processed/cleaned_recipes.parquet - run notebooks 01 and 02 first.")
df = pd.read_parquet(path)
print(f"Loaded {len(df)} recipes from {path.relative_to(ROOT)}")
print("canonical_ingredients present:", "canonical_ingredients" in df.columns)


Loaded 1983 recipes from data\processed\cleaned_recipes.parquet
canonical_ingredients present: True


In [2]:
from dish_groups import (build_groups, format_version_comparison, select_version,
                         version_facts, version_positions)

groups = build_groups(df)
multi = [g for g in groups if len(g.rows) > 1]
merged = sum(len(g.merged_rows) for g in groups)
print(f"{len(groups)} dish groups | {len(multi)} with multiple versions | {merged} near-identical rows merged")

for g in sorted(multi, key=lambda g: -len(g.rows))[:8]:
    print(f"  {len(g.rows)} versions: {g.display_name}")

1808 dish groups | 124 with multiple versions | 0 near-identical rows merged
  5 versions: Chicken Inasal
  4 versions: Pork Afritada
  4 versions: Chicken Afritada
  4 versions: Bicol Express
  4 versions: Dinuguan
  4 versions: Lechon Kawali
  4 versions: Crispy Pata
  4 versions: Pancit Malabon


In [3]:
# Reproduction of the three-version comparison, using the real Chicken Inasal rows.
target = [g for g in groups if g.name_key == "chicken inasal"]
assert target, "No 'chicken inasal' rows in this parquet snapshot"
group = target[0]

pantry = ["chicken", "vinegar", "garlic", "onion"]  # example pantry
eligible = version_positions(group, df, pantry)     # exactly the versions shown below
print(format_version_comparison(group, df, pantry))

We found 3 versions of Chicken Inasal that fit your ingredients:
A. Chicken Inasal - ~50 minutes (30 min active) | 8 missing (annatto, bay leaf, black pepper, brown sugar) | marinated; grilled
B. Chicken Inasal - ~35 minutes (30 min active) | 8 missing (bird's eye chili, calamansi, coconut, cooking oil) | marinated; grilled | adds bird's eye chili, coconut, cooking oil, soy sauce; skips annatto, bay leaf, black pepper, brown sugar vs A
C. Chicken Inasal - ~1 hour 18 minutes (18 min active) | 10 missing (annatto, black pepper, brown sugar, calamansi) | grilled; uses soda | adds lemon, margarine, soda; skips bay leaf vs A
Which version? Or say 'pick the fastest' / 'pick the one with fewest missing'.


In [4]:
# Selection stays in code; only the chosen row reaches the LLM context.
for label, choice in [("fastest", "fastest"), ("fewest missing", "fewest missing")]:
    pos = select_version(group, df, pantry, choice, eligible=eligible)
    row = df.iloc[pos]
    facts = version_facts(row, pantry)
    print(f"{label}: row {pos} | {row['recipe_name']} | {facts['display_time']} | "
          f"missing: {', '.join(facts['missing']) or 'none'}")
    print("  ingredients:", str(row["ingredients"])[:180])

fastest: row 458 | Chicken Inasal | 1 hour 18 minutes | missing: annatto, black pepper, brown sugar, calamansi, ginger, lemon, lemongrass, margarine, salt, soda
  ingredients: 2 lbs chicken cut into serving pieces | 2 tablespoons ginger minced | 2 tablespoons garlic minced | 3/4 cup lemongrass chopped | 1 cup coconut vinegar | 1/2 cup lemon or calamansi 
fewest missing: row 927 | Chicken Inasal | 50 minutes | missing: annatto, bay leaf, black pepper, brown sugar, calamansi, ginger, lemongrass, salt
  ingredients: 2 stalks lemongrass|1 head garlic, peeled and minced|1 thumb-size ginger, peeled and grated|1/2 cup palm vinegar|1/4 cup calamansi juice|1/4 cup brown sugar|1 1/2 teaspoons salt|1/


In [5]:
chosen = df.iloc[select_version(group, df, pantry, "fastest", eligible=eligible)]
context_block = (
    f"[Recipe] {chosen['recipe_name']}\n"
    f"Ingredients: {chosen['ingredients']}\n"
    f"Instructions: {str(chosen['instructions'])[:1200]}"
)
print(context_block[:900])

[Recipe] Chicken Inasal
Ingredients: 2 lbs chicken cut into serving pieces | 2 tablespoons ginger minced | 2 tablespoons garlic minced | 3/4 cup lemongrass chopped | 1 cup coconut vinegar | 1/2 cup lemon or calamansi juice | 1 tablespoon salt | 1/4 cup brown sugar | 1 cup lemon soda softdrink | 1/2 tablespoon ground black pepper | Basting Sauce: | 3 tablespoons annatto oil atsuete oil | 1/2 cup margarine softened | 1/4 teaspoon salt | 1 teaspoon lemon or calamansi juice
Instructions: In a freezer bag or large bowl, combine chicken, lemongrass, salt, ground black pepper, ginger, garlic, brown sugar, vinegar, lemon-lime soda, and lemon juice. | Stir or shake the mixture until every ingredient is well incorporated. Marinade for 1 to 3 hours. | In a bowl, combine margarine, annatto oil, salt, and lemon juice then stir. Set aside. | Grill the chicken while basting generous amount of the marga


`total_time` is shown as reported in the dataset; `active` minutes come from `cook_time_min`
and are what "I only have 15 minutes to cook" should be matched against. Datasets can disagree
between the two fields, so both are displayed.

## Integrating with notebook 04

1. Call `select_version(...)` after the user answers the comparison prompt.
2. Pass only that row to `format_context()` in `04_rag_pipeline.ipynb`.
3. Change `TOP_K` to count **dish groups** (via `rank_groups`) rather than rows, so two versions
   of the same name never occupy two recommendation slots.
4. At evaluation time, grade all versions of a name as equally relevant and report Recall@k
   and nDCG@k over groups.